# Data Vortex — Phase 2: SQL Challenge 10
## Month-over-Month Volume Growth & Cumulative Tracking

### 1. Challenge Description
Analyze monthly publishing volume across the 12-month dataset period (`2024-05` to `2025-04`), compute month-over-month (MoM) volume changes and percentage growth using `LAG()`, and track cumulative publishing volume using windowed `SUM()`.

In [ ]:
import os
import sqlite3
import pandas as pd

# File Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SQL_PATH = os.path.join(BASE_DIR, "sql", "challenge_10_mom_volume_growth.sql")

print(f"Target Database: {DB_PATH}")
print(f"SQL Script:      {SQL_PATH}")

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to SQLite database successfully.")

### 2. 12-Month Publishing Growth & Cumulative Volume Query
Applies `LAG(post_count) OVER (ORDER BY month)` and windowed `SUM()` to track monthly and cumulative trajectory.

In [ ]:
q_monthly = """
WITH monthly_base AS (
    SELECT 
        strftime('%Y-%m', timestamp) AS month,
        COUNT(post_id) AS post_count
    FROM posts
    GROUP BY strftime('%Y-%m', timestamp)
),
monthly_lag AS (
    SELECT 
        month,
        post_count,
        LAG(post_count) OVER (ORDER BY month) AS previous_month_post_count,
        SUM(post_count) OVER (
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_post_count
    FROM monthly_base
)
SELECT 
    month,
    post_count,
    previous_month_post_count,
    (post_count - previous_month_post_count) AS mom_change,
    CASE
        WHEN previous_month_post_count IS NULL OR previous_month_post_count = 0
        THEN NULL
        ELSE ROUND(
            100.0 * (post_count - previous_month_post_count) 
            / previous_month_post_count, 
            2
        )
    END AS mom_growth_pct,
    cumulative_post_count
FROM monthly_lag
ORDER BY month ASC;
"""

df_monthly = pd.read_sql_query(q_monthly, conn)
df_monthly

### 3. Summary Statistics Query
Aggregates total post volume, peak/trough months, and largest positive/negative growth rates.

In [ ]:
q_summary = """
WITH monthly_base AS (
    SELECT 
        strftime('%Y-%m', timestamp) AS month,
        COUNT(post_id) AS post_count
    FROM posts
    GROUP BY strftime('%Y-%m', timestamp)
),
monthly_trends AS (
    SELECT 
        month,
        post_count,
        LAG(post_count) OVER (ORDER BY month) AS previous_month_post_count,
        CASE
            WHEN LAG(post_count) OVER (ORDER BY month) IS NULL 
                 OR LAG(post_count) OVER (ORDER BY month) = 0
            THEN NULL
            ELSE ROUND(
                100.0 * (post_count - LAG(post_count) OVER (ORDER BY month)) 
                / LAG(post_count) OVER (ORDER BY month), 
                2
            )
        END AS mom_growth_pct
    FROM monthly_base
)
SELECT 
    (SELECT SUM(post_count) FROM monthly_base) AS total_posts,
    (SELECT month FROM monthly_base ORDER BY post_count DESC LIMIT 1) AS highest_volume_month,
    (SELECT post_count FROM monthly_base ORDER BY post_count DESC LIMIT 1) AS highest_volume_post_count,
    (SELECT month FROM monthly_base ORDER BY post_count ASC LIMIT 1) AS lowest_volume_month,
    (SELECT post_count FROM monthly_base ORDER BY post_count ASC LIMIT 1) AS lowest_volume_post_count,
    (SELECT month FROM monthly_trends WHERE mom_growth_pct IS NOT NULL ORDER BY mom_growth_pct DESC LIMIT 1) AS largest_positive_mom_month,
    (SELECT mom_growth_pct FROM monthly_trends WHERE mom_growth_pct IS NOT NULL ORDER BY mom_growth_pct DESC LIMIT 1) AS largest_positive_mom_growth_pct,
    (SELECT month FROM monthly_trends WHERE mom_growth_pct IS NOT NULL ORDER BY mom_growth_pct ASC LIMIT 1) AS largest_negative_mom_month,
    (SELECT mom_growth_pct FROM monthly_trends WHERE mom_growth_pct IS NOT NULL ORDER BY mom_growth_pct ASC LIMIT 1) AS largest_negative_mom_growth_pct;
"""

df_summary = pd.read_sql_query(q_summary, conn)
df_summary

### 4. Validation Checks
Verifies data integrity across all 11 Challenge 10 requirements.

In [ ]:
# Validation 1: Database contains exactly 12,000 posts
db_posts = conn.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
print(f"1. Posts in DB:               {db_posts} (Expected: 12000) -> {'PASS' if db_posts == 12000 else 'FAIL'}")

# Validation 2: Exactly 12 months returned
num_months = len(df_monthly)
print(f"2. Returned months:           {num_months} (Expected: 12) -> {'PASS' if num_months == 12 else 'FAIL'}")

# Validation 3: Monthly counts sum to 12,000
sum_monthly = df_monthly['post_count'].sum()
print(f"3. Sum of monthly counts:     {sum_monthly} (Expected: 12000) -> {'PASS' if sum_monthly == 12000 else 'FAIL'}")

# Validation 4: First month previous_month_post_count is NULL
first_prev_null = pd.isna(df_monthly['previous_month_post_count'].iloc[0])
print(f"4. First month prev is NULL:  {first_prev_null} (Expected: True) -> {'PASS' if first_prev_null else 'FAIL'}")

# Validation 5: First month mom_change is NULL
first_change_null = pd.isna(df_monthly['mom_change'].iloc[0])
print(f"5. First month change is NULL:{first_change_null} (Expected: True) -> {'PASS' if first_change_null else 'FAIL'}")

# Validation 6: First month cumulative equals post_count
first_cum_match = df_monthly['cumulative_post_count'].iloc[0] == df_monthly['post_count'].iloc[0]
print(f"6. First cumulative match:    {first_cum_match} (Expected: True) -> {'PASS' if first_cum_match else 'FAIL'}")

# Validation 7: Last month cumulative equals 12,000
last_cum_match = df_monthly['cumulative_post_count'].iloc[-1] == 12000
print(f"7. Last cumulative is 12000:  {last_cum_match} (Expected: True) -> {'PASS' if last_cum_match else 'FAIL'}")

# Validation 8: Cumulative counts never decrease (monotonically non-decreasing)
is_monotonic = df_monthly['cumulative_post_count'].is_monotonic_increasing
print(f"8. Monotonic cumulative:      {is_monotonic} (Expected: True) -> {'PASS' if is_monotonic else 'FAIL'}")

# Validation 9: Database and Cleaned CSV remain unchanged
df_clean = pd.read_csv(os.path.join(BASE_DIR, "data", "cleaned", "Social_Engine_Posts_Cleaned.csv"))
csv_match = len(df_clean) == 12000
print(f"9. DB & Cleaned CSV match:    {csv_match} (Expected: True) -> {'PASS' if csv_match else 'FAIL'}")

In [ ]:
# Close connection
conn.close()
print("Database connection closed cleanly. Challenge 10 Complete!")